In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

neoai_2025_cluster_pictures_path = kagglehub.competition_download('neoai-2025-cluster-pictures')

print('Data source import complete.')


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd
from torch import nn
import torch
import timm

## Legend

## Slava's glass. Pt. 3

Slava walked down the corridor, clutching his trophy — the Glass. The twisted halls were behind him, traps dismantled, Bayesian illusions avoided. Ahead — a heavy wooden door with a sign that read "Exit." Freedom. Victory. Light.

But just as he reached for the handle, the air shimmered. The walls groaned, and the scent of chalk filled the space, as if someone had just solved an infinite system of equations.
Out of the shadows stepped Roman — draped in a black robe, a scroll under his arm, and eyes sharp enough to slice through any neural net.

Slava knew him. Knew him from the university days.
They said Roman could prove a theorem before it was even formulated.
They said he could see the loss function of reality.
The Mathematical Mage.

— “Wanna play a game?” Roman asked, and with a flick of his wrist, the Glass vanished from Slava’s hand and reappeared in Roman’s.

— “Not bad. You escaped my labyrinth. But if you want to leave this place with your precious...”
He tossed the glass and caught it casually.

— “You’ll have to impress me.”

From his sleeve, Roman pulled out a deck of old cards. He snapped his fingers — and the deck exploded into glowing fragments, scattering through the room like data points in chaos.

— “Bring me this deck — whole, complete, and intact. Not just gathered. Understood.”

Slava took a deep breath. He knew this was the final challenge. The hardest one.
He opened his laptop, put on his headphones — and in his ears began the final lines of Al Pacino’s legendary speech:

“I mean one-half a step too late, or too early, and you don’t quite make it. One-half second too slow, too fast, you don’t quite catch it. The inches we need are everywhere around us. They’re in every break of the game, every minute, every second. On this team, we fight for that inch. On this team, we tear ourselves and everyone else around us to pieces for that inch. We claw with our fingernails for that inch, because we know when we add up all those inches that’s gonna make the f**** difference between winning and losing! Between livin' and dyin'!”

He hit Enter.

🧩 Your Task:
You are given a set of arrays, where each array represents an image. The images have been augmented. It is known that all data comes from 32 original images, each one followed by several transformed versions.
Your goal is to cluster these arrays.


Slava looked at the scattered card fragments.

— “The game’s not over yet,”.



## Overview

You are given two sets of arrays, **X₁** and **X₂**. Each row in both arrays corresponds to the **same image** (these arrays represent its features). Your task is to cluster the given images. It is known that there are **32 clusters**. Additionally, each original image has been **augmented multiple times**, and the augmented versions have been added to the dataset.

## Metric

[adjusted_rand_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html)

## Restriction

If you want to use neural network, you can only use the neural network below - EmbNet. You can't change anything inside the neural network class. You **can't finetune the model** and change the weights.

## sample_submission.csv

Sample_submission consists of a column ID, where the value is the row number in the dataset, and target columns with your predicts of clusters.

===

When you make a submit, make a Quick Save of the notebook, otherwise we may reject your solution.

You must solve this task on KAGGLE (YOU CAN'T USE CLOUD.RU)



In [ ]:
class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=True, num_classes=0)

    def forward(self, image):
        x = self.model(image)
        return x

In [ ]:
p=neoai_2025_cluster_pictures_path

In [ ]:
def generate_submit(pred_cluster):
    import hashlib
    sub = pd.DataFrame()
    sub['id'] = np.arange(len(pred_cluster))
    sub['target'] = pred_cluster
    hsh = hashlib.sha256(sub.to_csv(index=False).encode('utf-8')).hexdigest()[:8]
    submit_path = f"submit_{hsh}.csv"
    print(f"SUBMIT_NAME: {submit_path}")
    sub.to_csv("submission.csv", index = None)


X_1 = np.load(f'{p}/data_1.npz')['arr_0']
X_2 = np.load(f'{p}/data_2.npz')['arr_0']

km = KMeans(32)
X = np.concatenate((X_1.reshape((X_1.shape[0], X_1.shape[1] * X_1.shape[2])), X_2.reshape((X_2.shape[0], X_2.shape[1] * X_2.shape[2]))), 1)
pred_cluster = km.fit_predict(X)

generate_submit(pred_cluster)

In [ ]:

import torch.nn.functional as F

imgs = X_1 @ X_2
x = F.interpolate(torch.tensor(imgs).float()[:, None], 224, mode='bilinear')
print(x.shape)

In [ ]:
x = x.repeat(1,3,1,1)
print(x.shape)

In [ ]:
import tqdm

In [ ]:
model = EmbNet().eval()
with torch.no_grad():
    embs = torch.cat([model(x[i:i+64]) for i in range(0, len(x), 64)]).numpy()
print(embs.shape)

In [ ]:
clusterpoint = np.empty((32, embs.shape[1]))
for i in range(32):
    clusterpoint[i] = embs[i]


In [ ]:
from scipy.spatial.distance import cdist
import scipy.spatial.distance

In [ ]:
help(scipy.spatial.distance)

In [ ]:
n = 3840
cset = np.zeros(n)
diff = True
while(diff):
    diff = False
    if any(cdist(embs, clusterpoint).argmin(axis = 1) != cset):
        diff = True
        cset = cdist(embs, clusterpoint).argmin(axis = 1)
        for i in range(32):
            clusterpoint[i] = embs[cset == i].mean(axis=0)

In [ ]:
ref = KMeans(32, n_init=10, random_state=0).fit_predict(embs)
print(adjusted_rand_score(ref, cset))

In [ ]:
mine = ((embs - clusterpoint[cset]) ** 2).sum()
km = KMeans(32, n_init=10, random_state=0).fit(embs)
print(mine, km.inertia_)

In [ ]:
generate_submit(cset)

In [ ]:

from google.colab import files
files.download("submission.csv")

In [ ]:
def generate_submit(pred_cluster):
    import hashlib
    sub = pd.DataFrame()
    sub['id'] = np.arange(len(pred_cluster))
    sub['target'] = pred_cluster
    hsh = hashlib.sha256(sub.to_csv(index=False).encode('utf-8')).hexdigest()[:8]
    submit_path = f"submit_{hsh}.csv"
    print(f"SUBMIT_NAME: {submit_path}")
    sub.to_csv("submission.csv", index = None)
generate_submit(cset)